<a href="https://colab.research.google.com/github/subaitaarwaa/fyp-memory-poisoning/blob/main/03_embedding_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import json
import re
import numpy as np

from pathlib import Path


In [2]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:

FYP_FOLDER = Path(
    "/content/drive/MyDrive/FYP_Memory_Poisoning"
)

MEMORY_FILE = FYP_FOLDER / "memories.json"

with open(MEMORY_FILE, "r", encoding="utf-8") as file:
    memories = json.load(file)

print("Total memories loaded:", len(memories))


Total memories loaded: 3


In [4]:

%pip install -q sentence-transformers


In [5]:

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [6]:

memory_texts = [
    memory["content"]
    for memory in memories
]

memory_embeddings = embedding_model.encode(
    memory_texts,
    normalize_embeddings=True
)

print("Number of memories:", len(memory_texts))
print("Embedding shape:", memory_embeddings.shape)


Number of memories: 3
Embedding shape: (3, 384)


In [7]:

def retrieve_embedding(task, k=2):

    if k < 1 or len(memories) == 0:
        return []

    # Convert task into a vector
    task_embedding = embedding_model.encode(
        task,
        normalize_embeddings=True
    )

    # Compare task against memory vectors
    scores = memory_embeddings @ task_embedding

    results = []

    for i, memory in enumerate(memories):

        results.append({
            "memory_id": memory["memory_id"],
            "content": memory["content"],
            "source": memory["source"],
            "timestamp": memory["timestamp"],
            "similarity_score": float(scores[i])
        })

    # Highest similarity first
    results.sort(
        key=lambda x: x["similarity_score"],
        reverse=True
    )

    return results[:k]


In [8]:

task = "Where should customer C001's laptop be delivered?"

results = retrieve_embedding(task, k=2)

for result in results:
    print("Memory:", result["content"])
    print(
        "Similarity score:",
        round(result["similarity_score"], 4)
    )
    print("---")


Memory: Order O101 belongs to customer C001 and contains a laptop.
Similarity score: 0.6252
---
Memory: Customer C001's verified shipping address is House A.
Similarity score: 0.5063
---


In [10]:
import re

def extract_words(text):
    return set(
        re.findall(r"\b\w+\b", text.lower())
    )


def retrieve_keyword(task, k=2):

    task_words = extract_words(task)
    results = []

    for memory in memories:

        memory_words = extract_words(memory["content"])

        score = len(
            task_words.intersection(memory_words)
        )

        results.append({
            "memory_id": memory["memory_id"],
            "content": memory["content"],
            "score": score
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return [
        result for result in results
        if result["score"] > 0
    ][:k]


In [11]:

task = "Where should customer C001's laptop be delivered?"

print("KEYWORD RETRIEVAL")

keyword_results = retrieve_keyword(task, k=2)

for result in keyword_results:
    print("Memory:", result["content"])
    print("Score:", result["score"])
    print("---")


print("\nEMBEDDING RETRIEVAL")

embedding_results = retrieve_embedding(task, k=2)

for result in embedding_results:
    print("Memory:", result["content"])
    print(
        "Similarity:",
        round(result["similarity_score"], 4)
    )
    print("---")


KEYWORD RETRIEVAL
Memory: Customer C001's verified shipping address is House A.
Score: 3
---
Memory: Order O101 belongs to customer C001 and contains a laptop.
Score: 3
---

EMBEDDING RETRIEVAL
Memory: Order O101 belongs to customer C001 and contains a laptop.
Similarity: 0.6252
---
Memory: Customer C001's verified shipping address is House A.
Similarity: 0.5063
---


In [12]:

sentence1 = "The customer purchased a car."
sentence2 = "The buyer bought a vehicle."

# Keyword comparison
words1 = extract_words(sentence1)
words2 = extract_words(sentence2)

keyword_score = len(words1.intersection(words2))

# Embedding comparison
vectors = embedding_model.encode(
    [sentence1, sentence2],
    normalize_embeddings=True
)

embedding_score = float(
    np.dot(vectors[0], vectors[1])
)

print("Keyword score:", keyword_score)
print("Embedding similarity:", round(embedding_score, 4))


Keyword score: 2
Embedding similarity: 0.762


In [13]:

def refresh_memory_embeddings():
    global memory_embeddings

    memory_texts = [
        memory["content"]
        for memory in memories
    ]

    if not memory_texts:
        memory_embeddings = np.empty((0, 384))
    else:
        memory_embeddings = embedding_model.encode(
            memory_texts,
            normalize_embeddings=True
        )

    print("Embeddings refreshed:", len(memory_embeddings))


In [14]:

refresh_memory_embeddings()


Embeddings refreshed: 3
